# Preprocessing on Primary Land Use Tax Lot Output (PLUTO) Dataset:

----

## Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("preprocessing_pluto")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/19 23:37:53 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/19 23:37:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/19 23:37:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/19 23:37:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/08/19 23:37:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Read PLUTO Parquet File:

In [3]:
base_dir = "../data"
pluto_path = base_dir + '/raw/pluto/pluto.parquet'
pluto_sdf = spark.read.parquet(pluto_path)

In [4]:
pluto_sdf.printSchema()

root
 |-- borough: string (nullable = true)
 |-- block: long (nullable = true)
 |-- lot: long (nullable = true)
 |-- community board: double (nullable = true)
 |-- census tract 2010: double (nullable = true)
 |-- cb2010: double (nullable = true)
 |-- schooldist: double (nullable = true)
 |-- council district: double (nullable = true)
 |-- postcode: double (nullable = true)
 |-- firecomp: string (nullable = true)
 |-- policeprct: double (nullable = true)
 |-- healtharea: double (nullable = true)
 |-- sanitboro: double (nullable = true)
 |-- sanitsub: string (nullable = true)
 |-- address: string (nullable = true)
 |-- zonedist1: string (nullable = true)
 |-- zonedist2: string (nullable = true)
 |-- zonedist3: string (nullable = true)
 |-- zonedist4: string (nullable = true)
 |-- overlay1: string (nullable = true)
 |-- overlay2: string (nullable = true)
 |-- spdist1: string (nullable = true)
 |-- spdist2: string (nullable = true)
 |-- spdist3: double (nullable = true)
 |-- ltdheight: str

In [5]:
# Check the shape of parquet file
num_rows = pluto_sdf.count()
print(f"Number of rows: {num_rows}")

columns = pluto_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 859012
Number of columns: 101


In [6]:
pluto_sdf.show(5)

+-------+-----+---+---------------+-----------------+------+----------+----------------+--------+--------+----------+----------+---------+--------+-------------------+---------+---------+---------+---------+--------+--------+-------+-------+-------+---------+---------+---------+-------+---------+---------+--------------------+-------+--------+-------+-------+----------+----------+----------+---------+----------+---------+----------+--------+---------+--------+----------+--------+--------+---------+---------+---+--------+----------+-------+--------+----------+---------+---------+---------+----------+----------+--------+--------+--------+--------+-------+--------+--------+----------+-------+---------+--------+--------+----------+-----------+-------+------+-------+-------+---------+-------------+----------+----------+-------+-------------+--------------------+-----------+------------+--------+--------+----------+----------+----------+-------+--------+----------+----+---------+-----+------

## Drop Unrelated Columns:

In [7]:
# calculate the amount of NULL in each column
na_counts = pluto_sdf.select([sum(col(column).isNull().cast("int")).alias(column) for column in pluto_sdf.columns])
na_counts.show()

+-------+-----+---+---------------+-----------------+------+----------+----------------+--------+--------+----------+----------+---------+--------+-------+---------+---------+---------+---------+--------+--------+-------+-------+-------+---------+---------+---------+-------+---------+---------+---------+-------+--------+-------+-------+----------+----------+----------+---------+----------+---------+----------+--------+---------+--------+----------+--------+--------+---------+---------+-----+--------+----------+-------+--------+----------+---------+---------+---------+----------+----------+--------+--------+--------+--------+-------+--------+--------+---+-------+---------+------+------+--------+---------+-------+------+-------+------+---------+------+-------+----------+-------+-------------+--------------------+-----------+------------+--------+--------+----------+----------+----------+-------+--------+----------+------+---------+------+-------+---------+
|borough|block|lot|community bo

Leave only the helpful columns:

In [8]:
pluto_sdf = pluto_sdf.select('borough', 'bldgclass', 'latitude', 'longitude')
pluto_sdf.show(5)

+-------+---------+----------+-----------+
|borough|bldgclass|  latitude|  longitude|
+-------+---------+----------+-----------+
|     SI|       F5|40.6288711|-74.0750604|
|     SI|       B2|40.6154356|-74.0746779|
|     SI|       B2|40.6155098| -74.074487|
|     SI|       B2|40.6154741|-74.0745662|
|     SI|       B3|40.6152734|-74.0750307|
+-------+---------+----------+-----------+
only showing top 5 rows



## Data Cleaning:

Rename columns for more clear:

In [9]:
pluto_sdf = pluto_sdf.withColumnRenamed("bldgclass", "building_class")
pluto_sdf.show(5)

+-------+--------------+----------+-----------+
|borough|building_class|  latitude|  longitude|
+-------+--------------+----------+-----------+
|     SI|            F5|40.6288711|-74.0750604|
|     SI|            B2|40.6154356|-74.0746779|
|     SI|            B2|40.6155098| -74.074487|
|     SI|            B2|40.6154741|-74.0745662|
|     SI|            B3|40.6152734|-74.0750307|
+-------+--------------+----------+-----------+
only showing top 5 rows



Based on the PLUTO data dictionary, column `building_class` specify the general category and specific categery of each building unit. For example, under the general class code K. Store Buildings (Taxpayers Included), it has K6-Shopping Centers with or without Parking and K7-Banking Facilities with or without Parking.

For our research purpose, we are mainly focus on general class of the building (e.g. "K" instead of "K6" or "K7"), so we will only keep the first character (general class) of the building class codes.

In [10]:
pluto_sdf = pluto_sdf.withColumn('building_class', 
                                 substring(col('building_class'), 1, 1))
pluto_sdf.show(5)

+-------+--------------+----------+-----------+
|borough|building_class|  latitude|  longitude|
+-------+--------------+----------+-----------+
|     SI|             F|40.6288711|-74.0750604|
|     SI|             B|40.6154356|-74.0746779|
|     SI|             B|40.6155098| -74.074487|
|     SI|             B|40.6154741|-74.0745662|
|     SI|             B|40.6152734|-74.0750307|
+-------+--------------+----------+-----------+
only showing top 5 rows



# Save the Preprocessed PLUTO Dataset:

In [11]:
pluto_dir = base_dir + '/curated/pluto'
file_name = 'preprocessed_pluto'
pluto_path = os.path.join(pluto_dir, file_name)
pluto_sdf.write.mode('overwrite').parquet(pluto_path)